# Sanity-check `data/olist.db`

Connects read-only (via `settings.read_only_uri`), checks integrity, and runs a few queries that touch every table so a broken load shows up immediately.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import sqlite3
import pandas as pd
from config import settings

conn = sqlite3.connect(settings.read_only_uri, uri=True)
print(settings.db_path)

/Users/mohammad/Directory/projects/project_2/NL2SQL/data/olist.db


In [2]:
# integrity + row counts per table
print(conn.execute("PRAGMA integrity_check").fetchone())

tables = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()]

pd.DataFrame(
    [(t, conn.execute(f"SELECT count(*) FROM {t}").fetchone()[0]) for t in tables],
    columns=["table", "rows"],
)

('ok',)


,table,rows
0,customers,99441
1,sellers,3095
2,product_category_translation,71
3,products,32951
4,orders,99441
5,order_items,112650
6,order_payments,103886
7,order_reviews,99224
8,geolocation,1000163
9,sqlite_stat1,24


In [3]:
# order status breakdown
pd.read_sql("SELECT order_status, count(*) AS n FROM orders GROUP BY order_status ORDER BY n DESC", conn)

,order_status,n
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [4]:
# multi-table join: top customer states by order count
pd.read_sql(
    """
    SELECT c.customer_state, count(*) AS n
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY n DESC
    LIMIT 5
    """,
    conn,
)

,customer_state,n
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045


In [5]:
# bridge-table join: top product categories by revenue (English names)
pd.read_sql(
    """
    SELECT t.product_category_name_english AS category, round(sum(oi.price), 2) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
    GROUP BY category
    ORDER BY revenue DESC
    LIMIT 10
    """,
    conn,
)

,category,revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


In [6]:
# confirm the read-only URI actually blocks writes
try:
    conn.execute("DELETE FROM orders")
    print("WARNING: write succeeded, read-only guarantee broken")
except sqlite3.OperationalError as e:
    print("write correctly blocked:", e)

write correctly blocked: attempt to write a readonly database
